# 04 — Fit Linear Models (Interaction & Total Effects)

Fits OLS linear models per gene to estimate:
1. **Interaction model** — decomposes double-KO effects into single-KO main effects plus an interaction term.
2. **Total-effect model** — treats each double-KO as a single group, capturing the combined effect directly.

Outputs coefficient and p-value matrices for downstream analysis.

In [ ]:
import re

import numpy as np
import pandas as pd
import scanpy as sc
import statsmodels.formula.api as smf
from tqdm import tqdm

from config import (
    ANNDATA_BIG_COMBO,
    ANNDATA_COMBO_SCREEN,
    GENE_PREFIX_COLUMNS,
    TEXT_DIR,
)

## Helper functions

In [ ]:
def fit_ols_per_gene(guide_df: pd.DataFrame, expr_df: pd.DataFrame,
                     gene_names: pd.Index, formula: str) -> pd.DataFrame:
    """Fit an OLS model for each gene and collect summary tables."""
    all_res = []
    for gene in tqdm(gene_names, desc="Fitting OLS"):
        tmp = guide_df.copy()
        tmp["y"] = expr_df[gene].values
        mod = smf.ols(formula=formula, data=tmp).fit()
        res = mod.summary2().tables[1].reset_index()
        res["respGene"] = gene
        all_res.append(res)
    return pd.concat(all_res, ignore_index=True)


def pivot_results(csv_path: str, coef_out: str, pval_out: str) -> None:
    """Read OLS results CSV, pivot into KO x gene matrices, and save."""
    all_res = pd.read_csv(csv_path, header=1)
    all_res.columns = ["KO", "coef", "std", "t", "pval", "int025", "int097", "respGene"]
    all_res = all_res[["KO", "coef", "pval", "respGene"]]

    exclude = ["", "Intercept", "S_score", "G2M_score", "n_genes", "mt_frac", "NaN"]
    all_res = all_res[all_res["KO"].notna()]
    all_res = all_res[all_res["KO"] != ""]
    all_res = all_res[~all_res["KO"].isin(exclude)]

    coef = all_res.pivot(index="KO", columns="respGene", values="coef")
    coef = coef.apply(pd.to_numeric, errors="coerce")
    coef.to_csv(coef_out, index=True)

    pvals = all_res.pivot(index="KO", columns="respGene", values="pval")
    pvals = pvals.apply(pd.to_numeric, errors="coerce")
    pvals.to_csv(pval_out, index=True)

    print(f"Saved coefficients → {coef_out}")
    print(f"Saved p-values    → {pval_out}")

## Section 1 — Load and prepare data

In [ ]:
ANNDATA_BIG_COMBO

In [ ]:
ANNDATA_COMBO_SCREEN

In [ ]:
adata = sc.read(ANNDATA_BIG_COMBO)
adata = adata[adata.obs["batch"] == "1", :]

cols_to_drop = []
for c in adata.obs.columns:
    vals = adata.obs[c].astype(str)
    if (vals == "0").all() or (pd.to_numeric(vals, errors="coerce").fillna(0) == 0).all():
        cols_to_drop.append(c)

adata.obs.drop(columns=cols_to_drop, inplace=True)
print(f"Removed {len(cols_to_drop)} all-zero columns.")

adata.obs.drop(columns="batch", inplace=True)
adata.write(ANNDATA_COMBO_SCREEN)
print(f"Saved {ANNDATA_COMBO_SCREEN}")

In [ ]:
adata = sc.read(ANNDATA_COMBO_SCREEN)
adata

In [ ]:
adata.obs.columns

## Section 2 — Build design matrices and formulas

In [ ]:
expressionMatrix = pd.DataFrame(adata.X, columns=adata.var_names, index=adata.obs.index)
guideMatrix = adata.obs[GENE_PREFIX_COLUMNS].copy()

# Reconstruct a per-cell Guide label from the GENE_*_ indicator columns
# (the original `Guide` obs column was dropped during the Big/Combo merge).
def _row_guide(row):
    active = [c for c in GENE_PREFIX_COLUMNS if row[c] == 1]
    return "-".join(active) if active else "GENE_CONTROL_"

adata.obs["Guide"] = adata.obs[GENE_PREFIX_COLUMNS].apply(_row_guide, axis=1)

k = pd.DataFrame(adata.obs["Guide"].value_counts())
k["GuideGroup"] = k.index
k = k.loc[k["GuideGroup"] != "GENE_CONTROL_", :]

In [ ]:
k_inter = k.copy()
k_inter["GuideGroup"] = k_inter["GuideGroup"].str.replace("-", "*")
formula_inter = "y ~ " + " + ".join(k_inter["GuideGroup"])

k_total = k.copy()
k_total["GuideGroup"] = k_total["GuideGroup"].str.replace("-", "and")
formula_total = "y ~ " + " + ".join(k_total["GuideGroup"])

print("Interaction formula:")
print(formula_inter, "...")
print()
print("Total-effect formula:")
print(formula_total, "...")

## Section 3 — Fit interaction model

In [ ]:
inter_csv = str(TEXT_DIR / "allRes_ComboScreen.csv")

allRes_inter = fit_ols_per_gene(guideMatrix, expressionMatrix, adata.var_names, formula_inter)
allRes_inter.to_csv(inter_csv, index=False)
print(f"Saved interaction results → {inter_csv}")
print(f"Shape: {allRes_inter.shape}")

In [ ]:
allRes_inter

## Section 4 — Fit total-effect model

For the total-effect model, double-KO cells get a dedicated indicator column. Their
single-KO columns are zeroed out so the double-KO coefficient captures the **total**
(not residual) effect of knocking out both genes simultaneously.

In [ ]:
k_doubles = k_total[~k_total["GuideGroup"].isin(GENE_PREFIX_COLUMNS)].copy()
k_doubles["Guide1"] = k_doubles["GuideGroup"].str.split("and").str[0]
k_doubles["Guide2"] = k_doubles["GuideGroup"].str.split("and").str[1]

guideMatrix_total = guideMatrix.copy()

for i in range(k_doubles.shape[0]):
    row = k_doubles.iloc[i]
    col = guideMatrix_total[row["Guide1"]] + guideMatrix_total[row["Guide2"]]
    col = (col == 2).astype(int)
    guideMatrix_total[row["GuideGroup"]] = col

singleKOs = guideMatrix_total[guideMatrix_total.sum(axis=1) < 3]
doubleKOs = guideMatrix_total[guideMatrix_total.sum(axis=1) == 3].copy()
doubleKOs[GENE_PREFIX_COLUMNS] = 0

allKOGuides = pd.concat([singleKOs, doubleKOs])
expressionMatrix_total = expressionMatrix.loc[allKOGuides.index, :]

print(f"Single-KO cells: {singleKOs.shape[0]}")
print(f"Double-KO cells: {doubleKOs.shape[0]}")
print(f"Total cells:     {allKOGuides.shape[0]}")

In [ ]:
total_csv = str(TEXT_DIR / "allRes_total_ComboScreen.csv")

allRes_total = fit_ols_per_gene(allKOGuides, expressionMatrix_total, adata.var_names, formula_total)
allRes_total.to_csv(total_csv, index=False)
print(f"Saved total-effect results → {total_csv}")
print(f"Shape: {allRes_total.shape}")

In [ ]:
allRes_total

## Section 5 — Pivot results into coefficient and p-value matrices

In [ ]:
pivot_results(
    csv_path=inter_csv,
    coef_out=str(TEXT_DIR / "Coefs_ComboScreen.csv"),
    pval_out=str(TEXT_DIR / "Pvals_ComboScreen.csv"),
)

pivot_results(
    csv_path=total_csv,
    coef_out=str(TEXT_DIR / "Coefs_total_ComboScreen.csv"),
    pval_out=str(TEXT_DIR / "Pvals_total_ComboScreen.csv"),
)